# How does Qwen Performance on the SAT?

## Importing the Model

In [1]:
from llama_cpp import Llama

model_path = "/home/jovyan/shared/qwen2-1_5b-instruct-q4_0.gguf"

model = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

print("✅ Model loaded!")

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Model loaded!


In [2]:
def ask_sat_question(question, options):
    prompt = f"""
    Please reason step by step, and put your final answer within \\boxed{{}}.
    Here is an SAT-style multiple-choice question:
    
    Question: {question}
    Options:
    {options}

    Please provide your detailed reasoning and select the single best answer.
    """
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant that excels at academic questions."},
        {"role": "user", "content": prompt}
    ]
    
    response = model.create_chat_completion(
        messages=messages,
        max_tokens=512,
        temperature=0.6,
        top_p=0.95,
    )
    
    return response["choices"][0]["message"]["content"]

In [ ]:
# This cell don't work
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2-1.5B-Instruct" # This is the model we are using
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.generation_config.pad_token_id = tokenizer.pad_token_id

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

### Checkpoint #1
Let's take a step back and think critically about the model we chose to upload. Use these resources to help you answer the following question:

Article on Qwen 2-1.5
https://dataloop.ai/library/model/qwen_qwen2-15b/#:~:text=Performance%20The%20Qwen2%2D1.5B%20model%20is%20incredibly%20fast%2C,text%20classification%2C%20sentiment%20analysis%2C%20and%20language%20translation.

Article on Qwen 2-7
https://clarifai.com/qwen/qwenLM/models/qwen2-7b-chat

General Overview on Qwen Models
https://medium.com/@bgipradeep123/comparing-qwen-series-models-a-deep-dive-into-their-reasoning-capabilities-d8ee1bb0321e

In [4]:
# RUN THIS CELL TO ANSWER CHECKPOINT #1
user_response = input("Why do you think we imported Qwen2-1.5B compared to other Qwen models?\n")
file_path = 'answers.txt'
with open(file_path, 'w') as file:
    file.write(user_response)
    file.write('\n') 

Why do you think we imported Qwen2-1.5B compared to other Qwen models?
 Because it's smaller and requires less storage and memory


In [7]:
# I also overwrite this by Zoey
def ask_sat_question(question, options):
    prompt = f"""
    Please reason step by step, and put your final answer within \\boxed{{}}.
    Here is an SAT-style multiple-choice question:
    
    Question: {question}
    Options:
    {options}

    Please provide your detailed reasoning and select the single best answer.
    """
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant that excels at academic questions."},
        {"role": "user", "content": prompt}
    ]
    
    response = model.create_chat_completion(
        messages=messages,
        max_tokens=512,
        temperature=0.6,
        top_p=0.95,
    )
    
    return response["choices"][0]["message"]["content"]

This is an example of how we would ask Qwen to answer a sample question for us. Run it to see how Qwen responds!

In [8]:
question_text = "If 3x + 5 = 14, what is the value of x?"
options_text = "A) 2\nB) 3\nC) 4\nD) 5"

response = ask_sat_question(question_text, options_text)
print(response)

To solve this problem, we'll first isolate the variable x from the equation 3x + 5 = 14. Here's how we can do it:

1. Subtract 5 from both sides of the equation to get rid of the constant term on the left side:
   \(3x + 5 - 5 = 14 - 5\)
   \(3x = 9\)

2. Now, divide both sides of the equation by 3 to isolate x:
   \(\frac{3x}{3} = \frac{9}{3}\)
   \(x = 3\)

So, the value of x is 3. Therefore, the correct answer is B) 3.


### Using the SAT Questionbank

We can use a SAT Questionbank (linked here: https://pinesat.com/api/questions) to pick certain questions based on difficulty or domain. This can let us compare how the model performs depending on the type of "thinking" required.

The difficulty levels are "Easy", "Medium", or "Hard".

The available subjects in the question bank are...

    "Information and Ideas" — reading comprehension, main ideas
    "Standard English Conventions" — grammar, punctuation, sentence structure
    "Expression of Ideas" — rhetoric, author's purpose, transitions
    "Craft and Structure" — literary techniques, vocabulary in context

Take a look at the JSON file to get a sense of how the questionbank is structured: https://pinesat.com/api/questions

### Checkpoint #2

In [9]:
# RUN THIS CELL TO ANSWER CHECKPOINT #2
user_response = input("Why do you think we are testing the model with these specific domains? Why not math?\n")
file_path = 'answers.txt'
with open(file_path, 'a') as file:
    file.write(user_response)
    file.write('\n') 

Why do you think we are testing the model with these specific domains? Why not math?
 Because reading and writing questions are more subjective and require more context understanding


First, we will create a function to read the JSON file and fetch a random question for us, depending on the difficulty level and subject area.

In [10]:
import random, requests

def get_sat_question(difficulty = None, subject = None):
    questions = requests.get("https://pinesat.com/api/questions").json()
    if difficulty: questions = [q for q in questions if q["difficulty"].lower() == difficulty.lower()]
    if subject: questions = [q for q in questions if subject.lower() in q["domain"].lower()]
    return random.choice(questions) if questions else None

Here, we didn't specify a difficulty or domain subject. Take this as an example of how to use this function.

In [11]:
q = get_sat_question()
question = q["question"]["question"]
choices = q["question"]["choices"]
passage = q["question"]["paragraph"]
answer = q["question"]["correct_answer"]

# Some questions have a passage associated with them for the testtaker to read before answering the question.
# If that's the case, let's use that as the question instead
if passage and passage != "null":
    question = passage

In [ ]:
question

In [ ]:
choices

In [ ]:
passage

In [ ]:
question_text = question
options_text = choices

response = ask_sat_question(question_text, options_text)
print(response)

In [ ]:
answer

### Checkpoint #3

Now, you try! Write some code to pick a random Easy level question and have Qwen attempt the question. Did Qwen get the question right?

In [ ]:
### YOUR CODE HERE

question_text = ...
options_text = ...
response = ask_sat_question(question_text, options_text)
print(response)

In [ ]:
## Did Qwen get the question right? How can you tell? Write the code below.

### YOUR CODE HERE

Let's scale up and create a function to check if Qwen got the right answer automatically. This way, we can run several questions and see how Qwen performs overall.

In [ ]:
import re
def extract_answer(response, correct_answer):
    pattern = r"/correct\sanswer\sis\s(.+)\."
    a = re.search(pattern ,answer)
    if correct_answer in response:
        return 'CORRECT'
    else:
        return 'INCORRECT'

In [ ]:
extract_answer(response, answer)

### 4-Question Performance: Qwen

In [ ]:
q1 = get_sat_question('Easy', 'Information and Ideas')
q2 = get_sat_question('Easy', 'Standard English Conventions')
q3 = get_sat_question('Easy', 'Expression of Ideas')
q4 = get_sat_question('Easy', 'Craft and Structure')

q5 = get_sat_question('Medium', 'Information and Ideas')
q6 = get_sat_question('Medium', 'Standard English Conventions')
q7 = get_sat_question('Medium', 'Expression of Ideas')
q8 = get_sat_question('Medium', 'Craft and Structure')

q9 = get_sat_question('Hard', 'Information and Ideas')
q10 = get_sat_question('Hard', 'Standard English Conventions')
q11 = get_sat_question('Hard', 'Expression of Ideas')
q12 = get_sat_question('Hard', 'Craft and Structure')

question_bank = [q1, q2, q3, q4, q5, q6, q7, q8, q9, q10, q11, q12]

In [ ]:
import pandas as pd
df = pd.DataFrame(columns = ['Domain', 'Difficulty', 'Question', 'Choices', 'Response', 'Correct?'])
df

In [ ]:
for q in question_bank:
    question = q["question"]["question"]
    choices = q["question"]["choices"]
    passage = q["question"]["paragraph"]
    answer = q["question"]["correct_answer"]
    domain = q["domain"]
    difficulty = q['difficulty']
    if passage and passage != "null":
        question = passage

    # take the question
    response = ask_sat_question(question, choices)

    #check answer
    check = extract_answer(response, answer)

    #insert it all into a row in the dataframe
    new_row = {'Domain': domain, 'Difficulty': difficulty, 'Question': question, 'Choices': choices, 'Response' : response, 'Correct?': check}
    df.loc[len(df)] = new_row

In [ ]:
df

In [ ]:
pd.read_excel('Qwen_results.xlsx')

In [ ]:
with open('Qwen_results.xlsx', encoding="utf-8") as f:
    df = pd.read_excel(f)